In [ ]:
# capstone project -- function 5 (4D), week N

import numpy as np
import matplotlib.pyplot as plt

from itertools import product
from scipy.spatial import Delaunay, ConvexHull  # convex-hull check on the proposal
from scipy.stats import spearmanr, pearsonr     # model-free cross-check on the ARD verdict

from bayes_tools import (
    fitting,
    normalize, initial_bounds, validate_bounds_consistency,
    ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals, print_kappa_comparison,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 5

4D, 23 observations collected (20 initial + weeks 2-4). **The search domain is the unit cube `[0,1]^4`** — see below. All three collected points lie *outside* it, so **20 are used for modelling**.

Proposal this week: **`[0.224200, 0.846500, 1.000000, 1.000000]`** (GP mean 1527, std 157, predicted improvement **+438** on an incumbent of 1088.86).

## The domain is [0,1], and this notebook had been searching outside it

Every one of the 160+ initial X values across all eight functions lies inside `[0.0034, 0.9995]` — function 4's maximum is 0.999483, function 7's 0.998655, function 8's 0.998885. The initial designs were sampled from the unit cube, and `[0,1]^D` is the real domain. Bounds are now built with `initial_bounds(..., lower_limit=0.0, upper_limit=1.0)`, which gives exactly `[0,1]` on every axis.

This notebook was the worst affected. It searched **UCB restricted to x2** under `pad_fraction=1.0` bounds, which let x2 run to 1.67 while x0/x1/x3 stayed frozen:

| week | x2 | y | in domain? |
|---|---|---|---|
| 2 | 1.105882 | 6,146.78 | no |
| 3 | 1.365584 | 16,803.71 | no |
| 4 | 1.670022 | 57,792.24 | no |

`y` rose 9.4x then 3.4x, and `log10(y)` climbed a remarkably steady ~1.7 decades per unit x2. That looked like a spectacular ridge; it was the search walking out of the feasible region, with the black box happily evaluating its formula outside the range it was defined on. **Those three values cannot be the answer** — the best legitimate result on record is the initial batch's **1088.86**, not 57,792.

Week 4 also produced the error that surfaced this: its 6-dp submission `1.670022` rounded 4.8e-7 *above* x2's own padded bound (1.670021516812…), tripping `validate_bounds_consistency`. The proposal had been sitting exactly on the ceiling.

## Why the three collected points are excluded from the fit

They stay in the record. They are not used for modelling, for two reasons:

1. **They are unreachable.** Outside `[0,1]`, so no proposal can ever go there. A surrogate fitted to them spends its resolution on territory the search cannot use.
2. **They destroy resolution inside the cube.** With bounds = `[0,1]`:

| | x0 | x1 | x2 | x3 | y std |
|---|---|---|---|---|---|
| in-domain only (20) | 10.0 | 9.319 | 0.761 | **0.426** | 245.6 |
| all 23 | 10.0 | 10.0 | 0.629 | **10.0** | 12080.6 |

x3's length-scale inflates **23x** and pins at "irrelevant", and `y`'s std inflates **49x**. x3 is the *most* informative axis in-domain; including the out-of-domain points erases it.

Set `EXCLUDE_FROM_FIT = []` to revert. Note this discards all three collected evaluations from the fit — a real cost, accepted because the information they carry is largely redundant: the in-domain incumbent already sits at x2 = 0.8795, near the top of the cube, and the in-domain model independently wants to push x2 and x3 higher.

## What the in-domain model says

Length-scales `[10.0, 9.319, 0.761, 0.426]` against a domain width of 1.0 on every axis. So **x0 and x1 are flat** — their length-scales exceed the entire domain — and **x2 and x3 carry the signal**. The search holds x0/x1 at the incumbent and scans x2 and x3, the same construction functions 2 and 3 use for the same reason: a flat axis lets the optimiser park that coordinate at an arbitrary edge.

**The optimum appears to be on the domain boundary.** Every restricted search returns x2 = 1 and x3 = 1, at *every* `kappa` from 0.5 to 2.0:

| axes searched | proposal | pred. mean |
|---|---|---|
| x2 only | `[0.2242, 0.8465, 1.0, 0.8785]` | 1309.1 |
| x3 only | `[0.2242, 0.8465, 0.8795, 1.0]` | 1360.2 |
| **x2 and x3** | **`[0.2242, 0.8465, 1.0, 1.0]`** | **1527.1** |

**This changes how "on a bound" should be read here.** CLAUDE.md's rule is that upper-bound contact usually means `pad_fraction` is choosing the point rather than the model. That rule applies to *padded* bounds. These are the true domain edges, so a coordinate at 1.0 means the constrained optimum genuinely sits on the boundary — which is a legitimate answer, not an artifact. It also means `kappa` is nearly irrelevant on these axes: the response is monotone toward the corner inside the cube, so exploit, UCB and everything between pick the same point. `KAPPA = 1.0` is retained for continuity, not because the sweep discriminates.

## Notes

- **No y-scaling.** `kappa` is dimensionless. The in-domain `y` spread is 1088.7, so a raw `xi=0.01` would be meaningless for PI/EI — but nothing here uses them.
- **In-domain LOO**: R² +0.554, 65.8% of point pairs ordered correctly, 18/20 inside their own 95% interval. Modest but genuinely predictive.
- The proposal is only 0.121 and 0.122 past the observed x2/x3 maxima, well inside the domain — a small, legitimate step, not the kind of excursion the previous three weeks made.
- `plot_2d_bo` doesn't apply at D=4 — the GP view is `plot_nd_slices`.


In [18]:
X_initial = np.load("initial_data/function_5/initial_inputs.npy")
y_initial = np.load("initial_data/function_5/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 4, f"Expected a 4D problem, got {D}D input -- check the loaded file."

# Once, at the very start of the capstone:
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Include the new X and y values from the previous week, oldest first -- row order must be true chronological order, or `compute_iteration_diagnostics` at the bottom is meaningless.

One observation is recorded, and it is the new best by a factor of 5.65. Unlike function 4's, this point has clean provenance: the old sweep proposed exactly these coordinates, all positive, none on a bound.

In [ ]:
# Append last week's result BEFORE proposing this week's point, e.g.:
#
# new_X, new_y = append_observations(new_X, new_y, x_next, the_result_you_got)

new_X = np.array([
    [0.25621, 0.903056, 1.105882, 1.065351],    # week 2
    [0.25621, 0.903056, 1.365584, 1.065351],    # week 3 -- only x2 moved
    [0.25621 , 0.903056, 1.670022, 1.065351],
])

new_y = np.array([
    6146.77907747422,
    16803.705163569633,
    57792.23933083834,
])

# ---------------------------------------------------------------------------
# EXCLUDED FROM THE GP FIT: all three collected points.
#
# They stay in the record. They are not used for modelling because they lie
# OUTSIDE the search domain: the real domain is the unit cube [0,1]^4 (every
# initial X value across all eight functions is inside [0.0034, 0.9995]), and
# these have x2 up to 1.670 and x3 = 1.065. Under the old pad_fraction=1.0
# bounds the x2-restricted search was free to march out of the cube, which is
# exactly what it did for three weeks.
#
# Two reasons they must come out of the fit, not just be noted:
#   1. Unreachable -- no proposal inside [0,1] can go there, so fitting them
#      spends the surrogate's resolution on unusable territory.
#   2. They erase the informative axis. With bounds=[0,1], x3's length-scale
#      inflates 0.426 -> 10.0 (pinned "irrelevant", a 23x inflation) and y's
#      std inflates 245.6 -> 12080.6 (49x). x3 is the MOST informative axis
#      in-domain.
#
# Cost, stated plainly: this discards all three collected evaluations from the
# fit. Accepted because what they tell us is largely redundant -- the in-domain
# incumbent already sits at x2=0.8795 and the in-domain model independently
# wants x2 and x3 higher. Set EXCLUDE_FROM_FIT = [] to revert.
EXCLUDE_FROM_FIT = [0, 1, 2]

print("observations collected so far:", len(new_y),
      f"| excluded from the fit: {EXCLUDE_FROM_FIT}"
      f" -> {len(new_y) - len(EXCLUDE_FROM_FIT)} used")
print("latest y: %.3f -- but OUT OF DOMAIN, so not a legitimate result." % new_y[-1])
print("best legitimate (in-domain) y remains the initial batch's %.3f"
      % y_initial.max())


## Build the full dataset (initial + everything collected so far)

In [ ]:
# X_all / y_all: the complete record, for reporting only.
# X / y: the MODELLING set. Everything downstream uses X / y.
X_all, y_all = append_observations(X_initial, y_initial, new_X, new_y)

_use = np.ones(len(new_y), bool)
_use[EXCLUDE_FROM_FIT] = False
X, y = append_observations(X_initial, y_initial, new_X[_use], new_y[_use])

print("X_initial shape:", X_initial.shape, "| y_initial shape:", y_initial.shape)
print("combined X shape:", X.shape, "| combined y shape:", y.shape)

print("\nX range per dimension:")
print("  min:", np.round(X.min(axis=0), 4))
print("  max:", np.round(X.max(axis=0), 4))
print("y range: %.4f to %.4f (spread %.3f, std %.3f)"
      % (y.min(), y.max(), y.max() - y.min(), y.std()))

# THE DOMAIN IS THE UNIT CUBE. X is never negative (lower_limit=0.0) and never
# above 1 (upper_limit=1.0) -- every initial X value across all eight functions
# lies inside [0.0034, 0.9995]. Padding is clipped away entirely, so this is
# exactly [0,1] on each axis.
#
# Do NOT widen these bounds to enclose the excluded observations: that would put
# the search back outside the domain, which is the mistake being corrected.
bounds = initial_bounds(X_initial, pad_fraction=1.0,
                        lower_limit=0.0, upper_limit=1.0)
validate_bounds_consistency(X, bounds)      # modelling set only, all in-domain
print("\nBounds (the true domain):\n", np.round(bounds, 4))

print("\nexcluded points vs the domain:")
for i in EXCLUDE_FROM_FIT:
    out = [f"x{d}={new_X[i, d]:.4f}" for d in range(D) if new_X[i, d] > 1.0]
    print(f"  week {i + 2}: {', '.join(out)}  outside [0,1]   y={new_y[i]:.1f}")
print(f"  best OUT-of-domain y: {new_y.max():.1f}"
      f" | best IN-domain y: {y.max():.1f}"
      f"  ({new_y.max() / y.max():.0f}x, and not a legitimate result)")

# Kept as the record of how far out of the cube the search went. The old
# rationale here -- "each was the best result to date, so don't clamp to the
# observed region" -- is withdrawn: they were the best because they were outside
# the domain, and the search is now clamped to [0,1] by construction.
print("\ncollected points vs the INITIAL batch's range:")
for r in range(len(new_X)):
    n_out = 0
    for d in range(D):
        beyond = (new_X[r, d] > X_initial[:, d].max()
                  or new_X[r, d] < X_initial[:, d].min())
        n_out += beyond
        print(f"  week {r + 2}  x{d}={new_X[r, d]:.4f}  initial range"
              f" [{X_initial[:, d].min():.4f}, {X_initial[:, d].max():.4f}]"
              f"  outside={beyond}")
    print(f"  -> week {r + 2}: {n_out} of {D} coordinates outside,"
          f" y={new_y[r]:.4f}")

hull = Delaunay(X)
box = np.column_stack([X.min(axis=0), X.max(axis=0)])
box_volume = float(np.prod(box[:, 1] - box[:, 0]))
try:
    ch = ConvexHull(X)
    print(f"\nobserved bounding box volume: {box_volume:.4g}"
          f" | convex hull volume: {ch.volume:.4g} ({ch.volume / box_volume:.1%} of the box)")
    print(f"{len(ch.vertices)} of {len(X)} observations are convex-hull vertices")
except Exception as exc:  # degenerate point set -- not fatal
    print("\nconvex hull volume unavailable:", exc)

print(f"\nxi=0.01 would be {0.01 / (y.max() - y.min()):.7%} of the y spread -- meaningless."
      "\nUCB needs no correction: kappa is dimensionless. Any PI/EI xi below is raw-sized.")

## How badly did the model miss the new point, and how did it react?

Two things worth measuring before trusting any proposal.

First, refit on the initial batch alone and predict the collected point: this is a genuine out-of-sample test, since that fit never saw it. A large miss in units of its own predicted sigma means the model's uncertainty is untrustworthy *in that direction* -- and if the miss is upward, the surface climbs faster than the model believes, which argues for pushing further rather than backing off.

Second, compare the kernels before and after. One observation should not reshape the model; here it does, and the restrict-to-x2 construction below depends on the shape it settled into.

In [ ]:
# With all three collected points excluded, the modelling set IS the initial
# batch, so gp_before and gp_check coincide this week. Both are kept so the
# comparison reappears automatically once an in-domain observation lands.
with fitting("in-domain fit, plus a refit including the excluded points"):
    gp_before = fit_gp(X_initial, y_initial, bounds, n_restarts_optimizer=25,
                       random_state=0)
    gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=25, random_state=0)
    gp_outdom = fit_gp(X_all, y_all, bounds, n_restarts_optimizer=25, random_state=0)

print("length-scales, in-domain (live):", np.round(get_length_scales(gp_check), 3))
print("length-scales, incl. excluded  :", np.round(get_length_scales(gp_outdom), 3))
print("  -> x3 inflates to the pin; it is the most informative axis in-domain.")
print(f"y std: in-domain {y.std():.1f} | incl. excluded {y_all.std():.1f}\n")

# Fit is the initial 20 only, so every collected row is genuinely out of
# sample here. Note the model that actually PROPOSED week 3 also had week 2,
# so this understates what was known at proposal time for later rows.
mu_b, sd_b = gp_before.predict(normalize(new_X, bounds), return_std=True)
print("out-of-sample test on the collected points (fit = initial 20 only):")
for r in range(len(new_X)):
    miss = new_y[r] - mu_b[r]
    print(f"  week {r + 2}: predicted {mu_b[r]:+.1f} +/- {sd_b[r]:.1f}"
          f"   actual {new_y[r]:+.1f}")
    print(f"    {'UNDER' if miss > 0 else 'OVER'}-predicted by {abs(miss):.1f},"
          f" i.e. {miss / sd_b[r]:+.1f} sigma")
print("  -> (week 2 read) the model's uncertainty is far too tight in this")
print("     direction, and the error is upward, so its estimate of where the")
print("     ridge turns over is not to be taken at face value.")

print("\nkernel before:", gp_before.kernel_)
print("kernel after :", gp_check.kernel_)

ls_before, ls_after = get_length_scales(gp_before), get_length_scales(gp_check)
print("\nlength-scales before:", np.round(ls_before, 3))
print("length-scales after :", np.round(ls_after, 3))

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
print("\npinned (GP treats as irrelevant) before:",
      [d for d, v in enumerate(ls_before) if v >= 0.999 * PINNED] or "none")
print("pinned after                          :",
      [d for d, v in enumerate(ls_after) if v >= 0.999 * PINNED] or "none")

## Which axes actually move the response

Length-scales alone aren't enough here, for two separate reasons.

First, a length-scale pinned at the 10.0 ceiling means "smooth, nearly linear over this domain", which is *not* the same as "no effect" -- a mild monotone trend can still be present, and if it is, pushing along that axis is a real prediction rather than an optimiser artifact. So the cell below measures it directly: hold the other three axes at the incumbent, sweep each one, and record how much the GP mean moves. On that measure x2 dominates by more than 25x.

Second, and more awkwardly, **the ARD verdict is contested by the raw data.** The cell also computes model-free rank correlations, and they disagree:

| axis | spearman(y) | pearson(log y) | length-scale now | length-scale before the new point |
|---|---|---|---|---|
| x2 | +0.370 | +0.481 | 0.263 | 0.50 |
| **x3** | **+0.635** | **+0.656** | 9.07 ("irrelevant") | **0.25 (most informative)** |

So x3 has the *strongest* monotone association with `y` of any axis, and the pre-outlier GP thought it was the single most important one. Only the post-outlier fit calls it irrelevant, and that flip came from adding one observation.

Two of three lines of evidence therefore say x3 matters. The next section measures what including it would actually buy, and it is excluded on cost/benefit grounds rather than because it is irrelevant -- an important distinction if a future week's data shifts the balance.

Treat the GP-sweep numbers as "what the current model thinks" and the correlations as "what the data says without a model". When they disagree this sharply, neither should be taken as settled.

In [ ]:
incumbent = X[np.argmax(y)]
print("incumbent (best observed):", np.round(incumbent, 4), " y = %.3f" % y.max())
print("is the incumbent the largest observed value on each axis?",
      [bool(incumbent[d] == X[:, d].max()) for d in range(D)])

# --- What the current model thinks ---------------------------------------
print("\nGP mean variation along each axis (others held at the incumbent):")
print(f"{'axis':>5} | {'mean range, full bounds':>23} | {'mean range, observed':>20} | {'length-scale':>12}")
spans = np.zeros(D)
for d in range(D):
    row = []
    for lo, hi in [(bounds[d, 0], bounds[d, 1]), (X[:, d].min(), X[:, d].max())]:
        grid = np.tile(incumbent, (300, 1))
        grid[:, d] = np.linspace(lo, hi, 300)
        m, _ = gp_check.predict(normalize(grid, bounds), return_std=True)
        row.append(m.max() - m.min())
    spans[d] = row[0]
    print(f"   x{d} | {row[0]:23.1f} | {row[1]:20.1f} | {ls_after[d]:12.3f}")

# Which axes are worth searching? A length-scale at or beyond the DOMAIN WIDTH
# (1.0 on every axis, now that bounds are the unit cube) means the GP sees no
# structure along that axis across the whole feasible region -- the acquisition
# is flat there and its argmax is set by wherever the optimiser stops. Those get
# held at the incumbent; the rest are searched.
DOMAIN_W = bounds[:, 1] - bounds[:, 0]
FLAT = [d for d in range(D) if ls_after[d] >= DOMAIN_W[d]]
SEARCH_DIMS = tuple(d for d in range(D) if d not in FLAT)
DIM = int(np.argmin(ls_after))          # the single most informative axis

print(f"\nlength-scale vs domain width, per axis:")
for d in range(D):
    verdict = "FLAT (hold at incumbent)" if d in FLAT else "informative (search)"
    print(f"  x{d}: ls {ls_after[d]:7.3f} vs domain width {DOMAIN_W[d]:.3f}  -> {verdict}")
print(f"\nsearching {SEARCH_DIMS}; holding {tuple(FLAT)} at the incumbent."
      f" Most informative axis: x{DIM}")
if not SEARCH_DIMS:
    print("*** Every axis looks flat. The restricted construction is invalid -- stop. ***")

# --- What the data says without a model ----------------------------------
# Rank/log correlations are robust to the extreme magnitudes here, unlike the
# GP fit, which one observation can reshape. Where these disagree with the ARD
# length-scales, treat NEITHER as settled.
print("\nmodel-free association between each axis and y:")
print(f"{'axis':>5} | {'spearman(y)':>11} | {'pearson(log y)':>14} | {'ls now':>7} | {'ls before':>9}")
for d in range(D):
    sr = spearmanr(X[:, d], y)[0]
    pl = pearsonr(X[:, d], np.log(y))[0] if (y > 0).all() else np.nan
    print(f"   x{d} | {sr:+11.3f} | {pl:+14.3f} | {ls_after[d]:7.3f} | {ls_before[d]:9.3f}")

top_free = int(np.argmax([abs(spearmanr(X[:, d], y)[0]) for d in range(D)]))
if top_free != DIM:
    print(f"\n*** DISAGREEMENT: the GP sweep says x{DIM} dominates, but x{top_free} has the")
    print("    strongest model-free association. The next cell measures what including")
    print("    the contested axis in the search would actually buy. ***")

## Should the contested axis be searched too?

x3 is contested (see above), so rather than assume either way, measure what including it in the search would cost and buy. The cell below optimises UCB over several dimension subsets and reports, for each, the predicted mean and whether any coordinate lands on a bound.

The expected finding: every subset containing x3 pushes it to its **upper bound (1.8424) at every `kappa`** -- about 73% beyond the furthest x3 ever observed. What that extrapolation buys shrinks as `kappa` rises: roughly **+5.3%** predicted mean at `kappa=0`, **+1.4%** at the committed `kappa=1`, and it turns *negative* (-9.3%, -21.2%) by `kappa=2` and `kappa=3`. So at best it is a large extrapolation on a contested axis for a few percent, and at the operating point it is near-nothing. x3 stays fixed at the incumbent.

The reasoning that rules it out is the same one that rules out `kappa=5` below: when the acquisition runs to the edge of the box, `pad_fraction` is choosing the proposal rather than the model. Note that tightening `pad_fraction` does *not* fix this -- at 0.15, 0.25 and 0.5 all four coordinates end up on bounds, because the acquisition still wants the edge and the edge is simply closer. Restricting which axes are searched is the effective handle; shrinking the box is not.

If a future week's fit makes x3 non-pinned again, revisit this -- and give its upper bound explicit thought, because the acquisition will otherwise run straight to it.

In [23]:
def restricted_ucb(dims, kappa, n_grid=140):
    """Maximise UCB over `dims` only, holding every other axis at the incumbent."""
    axes = [np.linspace(bounds[d, 0], bounds[d, 1], n_grid) if d in dims
            else np.array([incumbent[d]]) for d in range(D)]
    grid = np.array(list(product(*axes)))
    gn = normalize(grid, bounds)
    m, s = gp_check.predict(gn, return_std=True)
    j = int(np.argmax(ucb_acquisition(gn, gp_check, kappa=kappa, maximize=True)))
    return grid[j], m[j], s[j]


def on_bound_dims(p):
    return [d for d in range(D)
            if np.isclose(p[d], bounds[d, 0]) or np.isclose(p[d], bounds[d, 1])]


print("UCB restricted to different dimension subsets, kappa=1.0\n")
print(f"{'search':>12} | {'proposal':>34} | {'pred mean':>10} | on bound")
base_mean = None
for dims in [(DIM,), (3,), (DIM, 3), (1, DIM, 3)]:
    p, m, s = restricted_ucb(dims, 1.0)
    if dims == (DIM,):
        base_mean = m
    print(f"{str(dims):>12} | {np.array2string(np.round(p, 4), separator=','):>34}"
          f" | {m:+10.1f} | {on_bound_dims(p) or 'none'}")

print(f"\nadding x3 to the search: x3 lands on its bound at EVERY kappa,")
print(f"and the predicted-mean gain over searching x{DIM} alone shrinks as kappa rises:")
for k in [0.0, 0.5, 1.0, 2.0, 3.0]:
    p, m, _ = restricted_ucb((DIM, 3), k)
    gain = 100 * (m - base_mean) / abs(base_mean)
    flag = "  <- committed kappa" if k == 1.0 else ""
    print(f"  kappa={k:<4g} x3={p[3]:.4f} (bound {bounds[3, 1]:.4f},"
          f" observed max {X[:, 3].max():.4f})  gain: {gain:+6.1f}%{flag}")

print(f"\nx3 at its bound is {100 * (bounds[3, 1] / X[:, 3].max() - 1):.0f}% beyond the"
      f" furthest x3 ever observed. At the committed kappa=1 that extrapolation buys")
print("about +1.4% predicted mean; by kappa=2 it is actively negative. A large")
print("extrapolation on a contested axis for near-nothing, so x3 stays at the")
print("incumbent -- excluded on cost/benefit, NOT on irrelevance.")

UCB restricted to different dimension subsets, kappa=1.0

      search |                           proposal |  pred mean | on bound
        (2,) |      [0.2562,0.9031,1.67  ,1.0654] |   +23248.7 | [2]
        (3,) |      [0.2562,0.9031,1.3656,1.8424] |   +16935.6 | [3]
      (2, 3) |      [0.2562,0.9031,1.67  ,1.8424] |   +23306.8 | [2, 3]
   (1, 2, 3) |      [0.2562,1.6869,1.67  ,1.8424] |   +23420.7 | [1, 2, 3]

adding x3 to the search: x3 lands on its bound at EVERY kappa,
and the predicted-mean gain over searching x2 alone shrinks as kappa rises:
  kappa=0    x3=1.8424 (bound 1.8424, observed max 1.0654)  gain:   +0.2%
  kappa=0.5  x3=1.8424 (bound 1.8424, observed max 1.0654)  gain:   +0.2%
  kappa=1    x3=1.8424 (bound 1.8424, observed max 1.0654)  gain:   +0.2%  <- committed kappa
  kappa=2    x3=1.8424 (bound 1.8424, observed max 1.0654)  gain:   +0.2%
  kappa=3    x3=1.8424 (bound 1.8424, observed max 1.0654)  gain:   +0.2%

x3 at its bound is 73% beyond the furthest x3 ever o

## Why clamping to the observed data would waste the week

Functions 3 and 4 both restrict or veto against the observed region, because extrapolation there was arbitrary or catastrophic. Function 5 is the opposite case, and this cell is the evidence rather than an assertion.

Optimise UCB along the dominant axis twice -- once free to reach the bounds, once clamped to the observed range -- at several `kappa`. If the clamped version keeps returning the incumbent's own coordinate, the mean is still climbing at the edge of the data and clamping would spend a real evaluation re-measuring a point already in hand.

In [24]:
print(f"UCB along x{DIM} only, x0/x1/x3 held at the incumbent"
      f" (incumbent x{DIM} = {incumbent[DIM]:.4f})\n")
print(f"{'kappa':>6} | {'free to bounds':>28} | {'clamped to observed':>28}")
for k in [0.0, 1.0, 2.0, 5.0]:
    cells = []
    for lo, hi in [(bounds[DIM, 0], bounds[DIM, 1]), (X[:, DIM].min(), X[:, DIM].max())]:
        grid = np.tile(incumbent, (2000, 1))
        grid[:, DIM] = np.linspace(lo, hi, 2000)
        gn = normalize(grid, bounds)
        m, s = gp_check.predict(gn, return_std=True)
        j = int(np.argmax(ucb_acquisition(gn, gp_check, kappa=k, maximize=True)))
        cells.append(f"x{DIM}={grid[j, DIM]:.4f} mean={m[j]:+8.1f}")
    print(f"{k:6g} | {cells[0]:>28} | {cells[1]:>28}")

print(f"\nThe clamped column returns the incumbent's own x{DIM} at every kappa:")
print("the mean has not yet turned over inside the sampled region, so clamping")
print("would propose a point we have already evaluated. Hence: extrapolate.")

UCB along x2 only, x0/x1/x3 held at the incumbent (incumbent x2 = 1.3656)

 kappa |               free to bounds |          clamped to observed
     0 |      x2=1.6700 mean=+23248.7 |      x2=1.3656 mean=+16793.4
     1 |      x2=1.6700 mean=+23248.7 |      x2=1.3656 mean=+16793.4
     2 |      x2=1.6700 mean=+23248.7 |      x2=1.3656 mean=+16793.4
     5 |      x2=1.6700 mean=+23248.7 |      x2=1.3656 mean=+16793.4

The clamped column returns the incumbent's own x2 at every kappa:
the mean has not yet turned over inside the sampled region, so clamping
would propose a point we have already evaluated. Hence: extrapolate.


## Backtest acquisition functions (using only data already collected)

Repeatedly splits the data into a "seed" set (fits the GP) and a held-out "candidate" set (true y known, hidden from the fit), then sees which config would have picked the best candidate most often. No new evaluations spent. `xi` values are in **raw `y`-units** (50 and 300), since a default 0.01 is 0.00016% of this spread.

**This metric is biased toward exploitation** -- it scores recognition of points whose `y` is already known, rewarding whatever ranks closest to the posterior mean, with no credit for reducing uncertainty. On functions 2-4 that made it near-useless or actively misleading.

Which is exactly why the result here carries weight: expect it to favour **high-`kappa` UCB** and rank `exploit` last -- the inverse of function 4 -- i.e. it prefers exploration *despite* being rigged against it. The mechanism is visible in the data: the best point sits far from all the others, so mean-ranking misses it while variance-seeking finds it. Watch for several UCB rows achieving a **median regret of 0**, meaning they pick the genuinely best candidate more than half the time.

Still not decisive alone -- mean regret is inflated by the 6147 point sitting in the candidate pool -- but it agrees with the direct evidence rather than contradicting it.

In [ ]:
backtest_configs = [
    {"name": "ucb_k1",   "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",   "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",   "acquisition": "ucb", "kappa": 5.0},
    {"name": "ucb_k10",  "acquisition": "ucb", "kappa": 10.0},
    {"name": "ei_xi50",  "acquisition": "ei",  "xi": 50.0},
    {"name": "pi_xi50",  "acquisition": "pi",  "xi": 50.0},
    {"name": "exploit",  "acquisition": "exploit"},
    {"name": "max_var",  "acquisition": "max_variance"},
]

with fitting("acquisition backtest, 50 splits"):
    backtest_results = backtest_acquisitions(
        X, y, bounds, backtest_configs,
        n_repeats=50, seed_frac=0.5, maximize=True,
        gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
    )

print_backtest_summary(backtest_results)

plot_acquisition_backtest(backtest_results)
plt.show()

## Compare kappa values -- the deciding cell

Two views. First `compare_kappa_proposals`, which optimises over all four axes and is shown mainly to demonstrate the problem: every `kappa` parks the three near-flat axes at box corners (x0 at its *lower* bound, x1 and x3 at their upper ones). Those coordinates are optimiser artifacts, not predictions.

Then the restricted view along the dominant axis, which is what the choice is actually made from.

**`kappa=1.0` is committed, and the reason is length-scale distance, not the mean's peak.** The GP mean along x2 does peak near 1.2745, but that peak is a **kernel artifact rather than a landmark**: it sits 0.1687 past the last observation, which is only **0.38 of one length-scale** (0.263 normalised = 0.439 in raw x2 units), exactly where the extrapolated gradient gives way to reversion toward the prior mean of about 437. Model-free there is no turnover at all -- `y` is still rising at the largest x2 sampled (0.879 gave 1089, then 1.106 gave 6147). So "step just past the peak" would be calibrating against an artifact.

The real criterion is to stay within roughly one length-scale of the data, where the model still carries information. `kappa` 0 to 2 all land in the 1.27-1.46 band, i.e. 0.4 to 1.0 length-scales out; `kappa=1` sits mid-band.

`kappa=5` is rejected structurally rather than by taste: it puts x2 exactly on the upper bound (1.6700), so `pad_fraction` -- an arbitrary choice -- would be picking the proposal instead of the model.

Two honest caveats on how weak the underlying trend is. Log `y` against x2 gives **r-squared = 0.23**, and the ascent is driven almost entirely by the last two points (at x2=0.21, `y` is 0.113; at x2=0.108 it is 233). A log-linear fit extrapolates to about 1585 at x2=1.3656 against the GP's 7344 -- not a rival prediction, since it under-predicts the incumbent tenfold, but a reminder that both extrapolations are weak. And the backtest's preference for `kappa=5`/`kappa=10` is really expressing "go far from the data", which in a restricted search just means "hit the box edge"; it does not override the structural objection.

In [ ]:
with fitting("unrestricted kappa sweep"):
    kappa_rows = compare_kappa_proposals(X, y, bounds,
                                         kappa_values=[0.5, 1.0, 2.0, 5.0, 10.0],
                                         maximize=True, n_restarts=30, random_state=0)

print("=== unrestricted over all 4 axes (shown to demonstrate the problem) ===")
print_kappa_comparison(kappa_rows)
print("\nhow many coordinates sit on a bound in each unrestricted proposal?")
for row in kappa_rows:
    n_edge = sum(np.isclose(row["x_next"][d], bounds[d, 0])
                 or np.isclose(row["x_next"][d], bounds[d, 1]) for d in range(D))
    print(f"  kappa={row['kappa']:<5g} on_bound={n_edge}/{D}"
          f"  in_hull={bool(hull.find_simplex(row['x_next']) >= 0)}")

plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()

print(f"\n=== restricted to x{DIM} (what the choice is made from) ===")
scan_grid = np.tile(incumbent, (4000, 1))
scan_grid[:, DIM] = np.linspace(bounds[DIM, 0], bounds[DIM, 1], 4000)
scan_norm = normalize(scan_grid, bounds)
scan_mean, scan_std = gp_check.predict(scan_norm, return_std=True)

# Express each candidate's step in LENGTH-SCALES past the last observation --
# that, not the GP's apparent peak, is the criterion (see the markdown above).
ls_raw = ls_after[DIM] * (bounds[DIM, 1] - bounds[DIM, 0])
last_obs = X[:, DIM].max()
print(f"x{DIM} length-scale: {ls_after[DIM]:.4f} normalised = {ls_raw:.4f} raw;"
      f" last observation at x{DIM}={last_obs:.4f}\n")
print(f"{'kappa':>6} | {'x' + str(DIM):>9} | {'step':>8} | {'length-scales out':>17}"
      f" | {'pred mean':>10} | {'pred std':>9} | on bound")
for k in [0.0, 1.0, 2.0, 5.0]:
    j = int(np.argmax(ucb_acquisition(scan_norm, gp_check, kappa=k, maximize=True)))
    xv = scan_grid[j, DIM]
    on_b = np.isclose(xv, bounds[DIM, 1]) or np.isclose(xv, bounds[DIM, 0])
    print(f"{k:6g} | {xv:9.4f} | {xv - incumbent[DIM]:+8.4f} | {(xv - last_obs) / ls_raw:17.2f}"
          f" | {scan_mean[j]:+10.1f} | {scan_std[j]:9.1f} | {on_b}")

peak = scan_grid[int(np.argmax(scan_mean)), DIM]
print(f"\nGP mean peaks along x{DIM} at {peak:.4f} (mean {scan_mean.max():+.1f}) --"
      f" but that is only {(peak - last_obs) / ls_raw:.2f} length-scales past the last")
print(f"observation, i.e. where extrapolated gradient gives way to reversion toward")
print(f"the prior mean ({y.mean():.1f}). It is a kernel artifact, not a measured maximum:")
hi = np.argsort(X[:, DIM])[-3:]
print("the highest-x%d observations show y still rising -- %s" % (
    DIM, ", ".join(f"x{DIM}={X[i, DIM]:.3f} -> y={y[i]:.1f}" for i in hi)))

# How weak is the underlying trend, model-free?
if (y > 0).all():
    coef = np.polyfit(X[:, DIM], np.log(y), 1)
    r2 = pearsonr(X[:, DIM], np.log(y))[0] ** 2
    print(f"\nmodel-free check: log y ~ {coef[0]:.3f}*x{DIM} + {coef[1]:.3f},"
          f" r^2 = {r2:.3f} (weak)")
    print(f"  its extrapolation at the kappa=1 point: y ~"
          f" {np.exp(np.polyval(coef, 1.3656)):.0f}, vs the GP's ~7344.")
    print("  It under-predicts the incumbent tenfold, so it is not a rival prediction --")
    print("  just a reminder that every extrapolation here is on thin evidence.")

KAPPA = 1.0
print(f"\nusing kappa = {KAPPA} (dimensionless -- no y-scaling needed),"
      f" searching x{DIM} only")

## Propose the next point

UCB scanned over **all informative axes at once** (`SEARCH_DIMS`, chosen above by comparing each length-scale against the domain width), with the flat axes held at the incumbent because the acquisition is flat along them and would otherwise park them at an arbitrary edge.

Searching both informative axes rather than just the single most informative one matters here: x2 alone predicts 1309, x3 alone 1360, **both together 1527**. The one-axis version was leaving about 167 of predicted mean on the table.

**Two checks change meaning now that bounds are the true domain.**

- **A coordinate on a bound is no longer a red flag.** Under `pad_fraction` padding it meant the box was choosing the point rather than the model. `[0,1]` is the real domain, so a coordinate at 1.0 means the *constrained optimum sits on the boundary* — a legitimate answer. The response here is monotone toward the x2=x3=1 corner inside the cube, which is also why `kappa` barely matters: every value from 0.5 to 2.0 returns the same point.
- **A convex-hull `False` is expected and benign**, but for a different reason than before. It is not that "the ridge continues past the sampled region" — that argument belonged to the out-of-domain search and is withdrawn. It is simply that the domain corner lies outside the hull of 20 interior points, as it must. What matters is that the step is *small*: 0.121 and 0.122 past the observed x2/x3 maxima, both inside the domain.

In [ ]:
# Scan every informative axis jointly, holding the flat ones at the incumbent.
n_grid = 120
axes = [np.linspace(bounds[d, 0], bounds[d, 1], n_grid) if d in SEARCH_DIMS
        else np.array([incumbent[d]]) for d in range(D)]
grid = np.array(list(product(*axes)))
grid_norm = normalize(grid, bounds)

gp = gp_check
mu_g, sd_g = gp.predict(grid_norm, return_std=True)
j = int(np.argmax(ucb_acquisition(grid_norm, gp, kappa=KAPPA, maximize=True)))
x_next = grid[j].copy()
mu, sigma = mu_g[j], sd_g[j]

print(f"--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print(f"searched {SEARCH_DIMS} over a {n_grid}^{len(SEARCH_DIMS)} grid;"
      f" held {tuple(FLAT)} at the incumbent")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)
print(f"GP predicted mean: {mu:.6g}, predicted std: {sigma:.6g}")
print(f"current best in-domain y: {y.max():.6g}"
      f"  -> predicted improvement: {mu - y.max():+.6g}")
for d in SEARCH_DIMS:
    print(f"  step along x{d} from the incumbent: {x_next[d] - incumbent[d]:+.4f}"
          f"   (observed max {X[:, d].max():.4f}, domain max {bounds[d, 1]:.1f})")
print(f"flat axes unchanged from the incumbent:"
      f" {np.allclose(x_next[list(FLAT)], incumbent[list(FLAT)]) if FLAT else 'n/a'}")

# What the one-axis version would have given, so the gain is visible.
for dims in [(DIM,)] + ([SEARCH_DIMS] if SEARCH_DIMS != (DIM,) else []):
    ax = [np.linspace(bounds[d, 0], bounds[d, 1], n_grid) if d in dims
          else np.array([incumbent[d]]) for d in range(D)]
    g2 = np.array(list(product(*ax))); gn2 = normalize(g2, bounds)
    m2, _ = gp.predict(gn2, return_std=True)
    j2 = int(np.argmax(ucb_acquisition(gn2, gp, kappa=KAPPA, maximize=True)))
    print(f"  searching {str(dims):<8} -> pred mean {m2[j2]:8.1f}")

# --- Checks --------------------------------------------------------------
# NOTE: bounds are the TRUE DOMAIN [0,1], not padded. A coordinate on a bound
# therefore means the constrained optimum is on the boundary -- legitimate, not
# a pad_fraction artifact. The old warning here said the opposite and has been
# removed; what is worth flagging instead is a LOWER-bound contact, which would
# mean the acquisition wants to leave the feasible region downward.
on_upper = [d for d in range(D) if np.isclose(x_next[d], bounds[d, 1])]
on_lower = [d for d in range(D) if np.isclose(x_next[d], bounds[d, 0])]
print("\non the domain's UPPER edge:", on_upper or "none",
      "  (legitimate: the optimum can sit on the boundary)")
print("on the domain's LOWER edge:", on_lower or "none")
if on_lower:
    print("*** A coordinate is on the 0.0 floor -- the acquisition wants to leave")
    print("    the feasible region. Treat that proposal with suspicion. ***")

assert np.all(x_next >= 0.0) and np.all(x_next <= 1.0), \
    "proposal is outside the [0,1] domain -- that is the bug this week fixed"

outside_obs = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("dimensions beyond the observed range (but inside the domain):",
      outside_obs or "none")
print(f"inside the convex hull of the observations: "
      f"{bool(hull.find_simplex(x_next) >= 0)}"
      "   (False expected: a domain corner lies outside the hull of interior points)")

print(f"\ndistance from the incumbent: {np.linalg.norm(x_next - incumbent):.4f}")
print("nearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+10.2f}  X={np.round(X[i], 3)}")

## Visualise the GP and acquisition function via 1D slices

Each panel holds the other three dimensions fixed at the current best observed point and sweeps one dimension. Dotted line is the fixed centre, dashed red is the proposed `x_next`, green is UCB.

What to expect, and it makes the whole argument visible in one figure: the **x2 panel** shows a steep ridge that is still rising at the right-hand edge of the observations, with `x_next` placed just past the mean's peak. The **x0, x1 and x3 panels are nearly flat** by comparison -- the y-axis range on each is a few hundred against x2's several thousand. That flatness is why those three coordinates are pinned at the incumbent rather than optimised.

This is a *partial* view: it shows the GP along each axis near the best point, not interactions between dimensions.

In [ ]:
plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=ucb_acquisition,
    x_next=x_next,
    acq_kwargs={"kappa": KAPPA, "maximize": True},
)
plt.show()

## Sanity-check the surrogate model: leave-one-out calibration

Refits the GP once per observation, leaving it out, and predicts it from the rest. At D=4 this is the main way to judge the surrogate, since the fitted surface can't be inspected directly.

Expect the new best point to be the worst-predicted by a wide margin -- that is the 17-sigma miss quantified earlier, reappearing here. The rest of the points should mostly fall inside their intervals. The reading to take away is not "the model is broken" but "the model is badly calibrated specifically in the direction we are moving", which is an argument for treating its predicted `std` as a floor rather than an estimate.

In [ ]:
with fitting("leave-one-out calibration"):
    pred_mean, pred_std = loo_predictions(X, y, bounds,
                                          gp_kwargs={"n_restarts_optimizer": 20})

plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")

# The modelling set is the initial batch alone this week (all three collected
# points are out of domain and excluded), so the old `worst == n_initial` test
# had no row to point at. Report which point it is instead.
worst = int(np.argmax(np.abs(y - pred_mean)))
src = "initial batch" if worst < n_initial else f"collected #{worst - n_initial + 1}"
print(f"worst-predicted point: index {worst} ({src}) -> true {y[worst]:.4g},"
      f" predicted {pred_mean[worst]:.4g} (std {pred_std[worst]:.4g})")

## Iteration diagnostics

`compute_iteration_diagnostics` replays the ordered `X`/`y` to reconstruct what the acquisition value, GP hyperparameters, and domain-wide uncertainty were at each past proposal -- no persisted log involved.

It applies one acquisition setting to the whole history, taken from `KAPPA` so it can't drift from what the proposal used. The single recorded observation was proposed under the old sweep's `pi`/`xi=0.01`, not `ucb`/`kappa=1`, so its replayed acquisition value describes a decision never made that way -- treat that column as meaningless until the history is UCB throughout. The GP-hyperparameter and `domain_mean_std` columns are unaffected by the mismatch.

It also assumes row order is true chronological order. `plot_bo_diagnostics` is skipped (it hard-codes a 2D scatter panel and this is 4D); the trend plots below are dimension-agnostic and gated on having a few completed iterations.

In [ ]:
# The replay only sees the MODELLING set, which is the initial batch alone this
# week -- all three collected points are out of domain and excluded. So there is
# nothing to replay, and n_replayed is the honest count (not len(new_y)).
n_replayed = len(y) - n_initial
history = None

if n_replayed == 0:
    print(f"Nothing to replay: {len(new_y)} observation(s) collected, all"
          f" {len(EXCLUDE_FROM_FIT)} excluded from the fit as out of domain.")
    print(f"Best in-domain y so far: {y.max():.4f}")
    print("This section becomes meaningful once an in-domain evaluation lands.")
else:
    with fitting("iteration-diagnostics replay"):
        history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                                acquisition="ucb", kappa=KAPPA,
                                                maximize=True)
    print("Best y so far:", np.nanmax(history["y"]))
    print(f"completed iterations in the modelling set: {n_replayed}"
          f"  ({len(new_y)} collected, {len(new_y) - n_replayed} excluded)")

    if n_replayed >= 3:
        for plot_fn in (plot_convergence, plot_acquisition_decay,
                        plot_uncertainty_shrinkage, plot_step_distance):
            plot_fn(history)
            plt.show()
    else:
        print(f"\nOnly {n_replayed} replayed iteration(s) -- need at least 3 before")
        print("the trend plots say anything. Skipping them.")

## Raw diagnostic fields

In [31]:
if history is not None:
    print("y:", np.round(history["y"], 3))
    print("\niteration:", history["iteration"])
    print("\nacq_value (NaN = initial batch; see the caveat above):", history["acq_value"])
    print("\npred_mean at proposal time:", history["pred_mean"])
    print("\nlength_scale:", history["length_scale"])
    print("\ndomain_mean_std:", history["domain_mean_std"])

y: [6.4443000e+01 1.8301000e+01 1.1300000e-01 4.2110000e+00 2.5837100e+02
 7.8434000e+01 5.7572000e+01 1.0957200e+02 8.8480000e+00 2.3322400e+02
 2.4423000e+01 6.4420000e+01 6.3477000e+01 7.9729000e+01 3.5580700e+02
 1.0888600e+03 2.8867000e+01 4.5182000e+01 4.3161300e+02 9.9720000e+00
 6.1467790e+03 1.6803705e+04]

iteration: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 2.]

acq_value (NaN = initial batch; see the caveat above): [          nan           nan           nan           nan           nan
           nan           nan           nan           nan           nan
           nan           nan           nan           nan           nan
           nan           nan           nan           nan           nan
 1948.85333804 9200.323387  ]

pred_mean at proposal time: [          nan           nan           nan           nan           nan
           nan           nan           nan           nan           nan
           nan           nan           nan           nan      

In [ ]:
# The proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X. THIS notebook hit that failure at week 5: a 6-dp copy of the
# week-4 proposal (1.670022) rounded 4.8e-7 outside x2's own bound and tripped
# validate_bounds_consistency. Bounds are the unit cube now, so a coordinate at
# exactly 1.0 is the case to watch.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))